# Chapter 12: Merge Sort and Quicksort

The programs from Chapter 12 of *Data Structures and Algorithms in Python, Hands-On* by Aadhavan Muthurengan, in the book's order. Each code cell is one program; the same program is in this folder as a `.py` file, and its expected output is in the `.out` file with the same name. Benchmark timings differ on every computer and every run.

Run the cells from top to bottom (**Run > Run All Cells**). The programs import each other's files, so keep this notebook in its chapter folder.

## Merging

`merge_sort.py`

In [1]:
"""Merge sort: split in half, sort each half, merge the sorted halves."""


def merge(left, right):
    """Merge two sorted lists into one sorted list."""
    out, i, j = [], 0, 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:  # <= takes from the left on ties: stable
            out.append(left[i])
            i += 1
        else:
            out.append(right[j])
            j += 1
    out.extend(left[i:])  # one of these two is empty
    out.extend(right[j:])
    return out


def merge_sort(items):
    """Return a new sorted list. Always about n log2 n comparisons."""
    if len(items) <= 1:
        return list(items)
    mid = len(items) // 2
    return merge(merge_sort(items[:mid]), merge_sort(items[mid:]))

## Quicksort

`quicksort.py`

In [2]:
"""Quicksort: pick a pivot, put smaller items left of it and bigger
ones right of it, then sort each side. Sorts in place."""

import random


def partition(items, lo, hi):
    """Lomuto partition of items[lo..hi] around items[hi]. Returns the
    pivot's final index: everything left of it is smaller."""
    pivot = items[hi]
    store = lo
    for i in range(lo, hi):
        if items[i] < pivot:
            items[i], items[store] = items[store], items[i]
            store += 1
    items[store], items[hi] = items[hi], items[store]
    return store


def quicksort_naive(items, lo=0, hi=None):
    """Always uses the last item as the pivot."""
    if hi is None:
        hi = len(items) - 1
    if lo < hi:
        p = partition(items, lo, hi)
        quicksort_naive(items, lo, p - 1)
        quicksort_naive(items, p + 1, hi)


def quicksort(items, lo=0, hi=None):
    """A random pivot, and recursion only on the smaller side, so the
    stack stays about log2(n) deep even in an unlucky case."""
    if hi is None:
        hi = len(items) - 1
    while lo < hi:
        r = random.randint(lo, hi)
        items[r], items[hi] = items[hi], items[r]
        p = partition(items, lo, hi)
        if p - lo < hi - p:
            quicksort(items, lo, p - 1)
            lo = p + 1  # loop on the bigger side instead of recursing
        else:
            quicksort(items, p + 1, hi)
            hi = p - 1


def quicksort3(items, lo=0, hi=None):
    """Three-way partition: items equal to the pivot are gathered in the
    middle and never touched again. Fast when there are many duplicates."""
    if hi is None:
        hi = len(items) - 1
    while lo < hi:
        pivot = items[random.randint(lo, hi)]
        lt, i, gt = (
            lo,
            lo,
            hi,
        )  # [lo,lt) < pivot, [lt,i) == pivot, (gt,hi] >
        while i <= gt:
            if items[i] < pivot:
                items[lt], items[i] = items[i], items[lt]
                lt, i = lt + 1, i + 1
            elif items[i] > pivot:
                items[i], items[gt] = items[gt], items[i]
                gt -= 1
            else:
                i += 1
        if lt - lo < hi - gt:
            quicksort3(items, lo, lt - 1)
            lo = gt + 1
        else:
            quicksort3(items, gt + 1, hi)
            hi = lt - 1

`worst_case.py`

In [3]:
"""Count quicksort's comparisons on random and on sorted input.
With the last item as pivot, sorted input splits off one item at a
time: n * (n - 1) / 2 comparisons, and n levels of recursion."""

import math
import random
import sys

import quicksort as q

calls = {"compared": 0}
original = q.partition


def counting_partition(items, lo, hi):
    calls["compared"] += hi - lo  # the loop compares each item once
    return original(items, lo, hi)


q.partition = counting_partition
sys.setrecursionlimit(5000)  # the naive version recurses n levels deep
random.seed(2026)
n = 2000
for name, sort in [
    ("naive", q.quicksort_naive),
    ("random pivot", q.quicksort),
]:
    for label, data in [
        ("random", random.sample(range(n), n)),
        ("sorted", list(range(n))),
    ]:
        calls["compared"] = 0
        sort(data)
        count = calls["compared"]
        print(f"{name:>12}, {label} input: {count:>9,} comparisons")
print(f"n log2 n = {n * math.log2(n):,.0f}")
print(f"n (n - 1) / 2 = {n * (n - 1) // 2:,}")

       naive, random input:    24,259 comparisons
       naive, sorted input: 1,999,000 comparisons
random pivot, random input:    25,072 comparisons
random pivot, sorted input:    23,604 comparisons
n log2 n = 21,932
n (n - 1) / 2 = 1,999,000


## The Race

`bench_sorts.py`

In [4]:
"""Merge sort and quicksort against Python's sorted(), on random data."""

import random

import bench
from merge_sort import merge_sort
from quicksort import quicksort, quicksort3

rng = random.Random(2026)


def shuffled(n):
    return (rng.sample(range(n), n),)


sizes = [10_000, 20_000, 40_000, 80_000]
results = {
    "merge_sort": bench.doubling(merge_sort, shuffled, sizes, fresh=True),
    "quicksort": bench.doubling(quicksort, shuffled, sizes, fresh=True),
    "quicksort3": bench.doubling(quicksort3, shuffled, sizes, fresh=True),
    "sorted()": bench.doubling(sorted, shuffled, sizes, fresh=True),
}
bench.table(results)
bench.chart(results, "ch12_sorts.png", "Sorting random lists")

         n      merge_sort       quicksort      quicksort3        sorted()
    10,000         12.8 ms         9.12 ms         12.5 ms         1.24 ms
    20,000    26.7 ms x2.1    21.1 ms x2.3      25 ms x2.0    2.72 ms x2.2
    40,000    57.6 ms x2.2    41.5 ms x2.0    60.6 ms x2.4    6.12 ms x2.2
    80,000     131 ms x2.3     101 ms x2.4     130 ms x2.1    14.2 ms x2.3


`bench_duplicates.py`

In [5]:
"""Many equal items: the two-way and three-way quicksorts."""

import random
import sys

import bench
from quicksort import quicksort, quicksort3

sys.setrecursionlimit(10_000)
rng = random.Random(2026)


def few_values(n):
    return ([rng.randint(0, 9) for _ in range(n)],)  # only 10 values


sizes = [2_000, 4_000, 8_000, 16_000]
results = {
    "quicksort": bench.doubling(quicksort, few_values, sizes, fresh=True),
    "quicksort3": bench.doubling(quicksort3, few_values, sizes, fresh=True),
}
bench.table(results)

         n       quicksort      quicksort3
     2,000         4.32 ms          374 us
     4,000    16.8 ms x3.9     844 us x2.3
     8,000    60.4 ms x3.6     1.6 ms x1.9
    16,000     238 ms x3.9    3.02 ms x1.9


## How Python Sorts: Timsort

`bench_timsort.py`

In [6]:
"""Python's sorted() (Timsort) finds runs that are already in order."""

import random

import bench

rng = random.Random(2026)
n = 1_000_000
ordered = list(range(n))
nearly = list(range(n))
for _ in range(100):
    i, j = rng.randrange(n), rng.randrange(n)
    nearly[i], nearly[j] = nearly[j], nearly[i]
cases = {
    "random": rng.sample(range(n), n),
    "sorted": ordered,
    "reversed": ordered[::-1],
    "100 swaps": nearly,
}
for name, data in cases.items():
    seconds = bench.measure(lambda: sorted(data))
    print(f"{name:>10}: {bench.show_time(seconds)}")

    random: 333 ms


    sorted: 10 ms
  reversed: 14 ms


 100 swaps: 21.3 ms
